# 🎬 Remotion Video Export (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2chandankumarjha-ux/remotion-reel-export/blob/main/Remotion_Reel_Export.ipynb)

**Order:** 1 → 2 → 3 chalao (sirf pehli baar / runtime reset ke baad). Phir naya reel banane ke liye sirf **4 (GUI Paste)** aur **5 (Render)**.

Project ki saari files GitHub repo me hain, zip upload karne ki zarurat nahi.

## 1️⃣ System setup (Node + Chrome libraries + ffmpeg)

In [ ]:
%%bash
set -e
export DEBIAN_FRONTEND=noninteractive

sudo apt-get update -qq 2>&1 | grep -v '^W:' || true

PKGS="ca-certificates fonts-liberation wget xdg-utils ffmpeg libnss3 libdbus-1-3 libatk1.0-0 \
libatk-bridge2.0-0 libgbm1 libasound2 libxrandr2 libxkbcommon0 libxfixes3 libxcomposite1 \
libxdamage1 libpango-1.0-0 libcairo2 libcups2 libxss1 libxtst6"

has_candidate() {
  c=$(apt-cache policy "$1" 2>/dev/null | awk '/Candidate:/{print $2}')
  [ -n "$c" ] && [ "$c" != "(none)" ]
}

INSTALL=""
for p in $PKGS; do
  if has_candidate "$p"; then INSTALL="$INSTALL $p"
  elif has_candidate "${p}t64"; then INSTALL="$INSTALL ${p}t64"
  else echo "⚠️ skip (nahi mila): $p"
  fi
done
sudo apt-get install -y -qq $INSTALL > /dev/null

NODE_MAJOR=0
if command -v node >/dev/null 2>&1; then NODE_MAJOR=$(node -v | sed 's/v\([0-9]*\).*/\1/'); fi
if [ "$NODE_MAJOR" -lt 20 ]; then
  curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - > /dev/null
  sudo apt-get install -y -qq nodejs > /dev/null
fi

echo "✅ node $(node -v) | npm $(npm -v) | $(ffmpeg -version | head -1)"

## 2️⃣ Project clone (GitHub se)

In [ ]:
import os, shutil, subprocess

# ====== SETTINGS ======
REPO_URL = "https://github.com/2chandankumarjha-ux/remotion-reel-export"   # <-- apna repo link
BRANCH   = "main"
# ======================

PROJECT = "/content/my_project"

if "YOUR_USERNAME" in REPO_URL:
    raise SystemExit("❌ Pehle REPO_URL me apna GitHub repo link daalein.")

# Private repo ke liye optional token (Colab Secrets -> GITHUB_TOKEN)
token = None
url = REPO_URL
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    if token and url.startswith("https://github.com/"):
        url = url.replace("https://", f"https://{token}@", 1)
except Exception:
    pass   # public repo: token ki zarurat nahi

shutil.rmtree(PROJECT, ignore_errors=True)
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, PROJECT],
                   capture_output=True, text=True)
if r.returncode != 0:
    err = r.stderr or ""
    if token:
        err = err.replace(token, "***")      # token print na ho
    print(err[-1500:])
    raise SystemExit("❌ Clone fail hua. Repo link / branch / token check karein.")

for need in ["package.json", "src/index.jsx", "src/Root.jsx"]:
    if not os.path.exists(f"{PROJECT}/{need}"):
        raise SystemExit(f"❌ Repo me '{need}' nahi mili. Pehle saari files push karein.")

print("✅ Project ready:", PROJECT)
print(sorted(os.listdir(PROJECT)))

## 3️⃣ Dependencies install (ek hi baar)
Yahan `npm install` use kiya gaya hai taaki naye packages bina kisi error ke automatically install ho jayein.

In [ ]:
import json, os, subprocess

PROJECT = "/content/my_project"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, cwd=PROJECT, text=True, capture_output=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise SystemExit(f"❌ Fail: {cmd}")
    return r.stdout

# npm install use kar rahe hain taaki lockfile mismatch ka error na aaye
sh("npm install --no-audit --no-fund")

print("✅ Saari dependencies install ho gayi hain.")

## 4️⃣ GUI: Naya Root.jsx Paste Karein
**ChatGPT se naya reel banwane ke liye:** Neeche diye gaye example code ko copy karke ChatGPT ko dein aur bolein *"Is format me mujhe [Topic] par naya data reel banao, data aur colors change kardo"*.

**Code Save karne ke liye:** Neeche apna naya code paste karein, ek **Theme Name** daalein (jaise `dark_blue`), aur **Save** button dabayein. Ye code `Root.jsx` me save hoga aur ek backup `src/themes/` folder me bhi chala jayega.

In [ ]:
#@title ✏️ Naya Root.jsx Paste, Theme Save & Apply GUI
import ipywidgets as widgets
from IPython.display import display
import os

ROOT_PATH = "/content/my_project/src/Root.jsx"
THEMES_DIR = "/content/my_project/src/themes"

if not os.path.exists(THEMES_DIR):
    os.makedirs(THEMES_DIR)

# Example Code for ChatGPT
EXAMPLE_CODE = """import { AbsoluteFill, useCurrentFrame, useVideoConfig, spring, interpolate, Composition } from 'remotion';

// ==========================================
// 🤖 CONFIGURATION (Isko ChatGPT me paste karein)
// ==========================================
const CONFIG = {
  title: "Top 10 Population Countries",
  theme: {
    bg: "#1a1a1a",
    text: "#ffffff",
    primary: "#4CAF50",
    secondary: "#FFC107"
  },
  data: [
    { name: "India", value: 1428 },
    { name: "China", value: 1425 }
  ],
  captions: [
    { text: "Duniya ki aabaadi...", start: 0, end: 120 },
    { text: "India sabse aage!", start: 120, end: 300 }
  ]
};

// ==========================================
// 🎬 MAIN COMPONENT
// ==========================================
export const MyReel = () => {
  const frame = useCurrentFrame();
  const { fps } = useVideoConfig();

  const opacity = interpolate(frame, [0, 30], [0, 1]);
  const scale = spring({ frame, fps, config: { damping: 12 } });

  const currentCaption = CONFIG.captions.find(c => frame >= c.start && frame <= c.end);

  return (
    <AbsoluteFill style={{ backgroundColor: CONFIG.theme.bg, color: CONFIG.theme.text, fontFamily: 'sans-serif' }}>
      <h1 style={{ textAlign: 'center', marginTop: 100, opacity, transform: `scale(${scale})` }}>
        {CONFIG.title}
      </h1>

      <div style={{ padding: 50 }}>
        {CONFIG.data.map((item, i) => (
          <div key={i} style={{ marginBottom: 20 }}>
            <div style={{ display: 'flex', justifyContent: 'space-between' }}>
              <span>{item.name}</span>
              <span>{item.value}M</span>
            </div>
            <div style={{ height: 20, backgroundColor: '#333', borderRadius: 10 }}>
              <div style={{ height: '100%', width: `${(item.value / 1500) * 100}%`, backgroundColor: CONFIG.theme.primary, borderRadius: 10 }} />
            </div>
          </div>
        ))}
      </div>

      {currentCaption && (
        <div style={{ position: 'absolute', bottom: 100, width: '100%', textAlign: 'center' }}>
          <span style={{ backgroundColor: 'rgba(0,0,0,0.7)', padding: '10px 20px', borderRadius: 10, fontSize: 24 }}>
            {currentCaption.text}
          </span>
        </div>
      )}
    </AbsoluteFill>
  );
};

// ==========================================
// 🚀 REGISTRATION
// ==========================================
export const Root = () => (
  <Composition id="MyReel" component={MyReel} durationInFrames={300} fps={30} width={1080} height={1920} />
);
"""

# 1. Code Input
code_input = widgets.Textarea(
    value=EXAMPLE_CODE,
    placeholder='Yahan apna naya Root.jsx ka code paste karein...',
    description='Code:',
    layout=widgets.Layout(width='100%', height='400px')
)

# 2. Theme Name Input
theme_input = widgets.Text(
    value='my_custom_theme',
    placeholder='e.g. dark_reel',
    description='Theme Name:',
    layout=widgets.Layout(width='50%')
)

# 3. Save Button
save_btn = widgets.Button(description="💾 Save Root.jsx & Theme", button_style='success')
output_area = widgets.Output()

def save_code(b):
    with output_area:
        output_area.clear_output()
        code = code_input.value.strip()
        if not code:
            print("❌ Code khali hai. Pehle code paste karein.")
            return
        if not os.path.exists(ROOT_PATH):
            print("❌ File nahi mili. Kya aapne Cell 2 (Clone) chalaya?")
            return
        try:
            # Save main file
            with open(ROOT_PATH, "w", encoding="utf-8") as f:
                f.write(code + "\n")
            
            # Save theme backup
            theme_name = theme_input.value.strip() or "default_theme"
            theme_path = os.path.join(THEMES_DIR, f"{theme_name}.jsx")
            with open(theme_path, "w", encoding="utf-8") as f:
                f.write(code + "\n")

            print(f"✅ Root.jsx update ho gaya!")
            print(f"📁 Theme backup save ho gaya: src/themes/{theme_name}.jsx")
            print("Ab Cell 5 (Render) chalayein.")
        except Exception as e:
            print(f"❌ Error: {e}")

save_btn.on_click(save_code)

display(widgets.VBox([code_input, theme_input, save_btn, output_area]))

## 5️⃣ Render → Download
Settings badal kar chalayein. `ONLY_STILL = True` karoge to sirf ek frame (PNG) banega, jaldi check ke liye.

In [ ]:
#@title 🎬 Render → Download
import os, re, shutil, subprocess

# ====== SETTINGS ======
COMP_ID     = ""             # khali = Root.jsx se auto-detect (last composition)
OUTPUT_NAME = "Naya_Reel.mp4"
ONLY_STILL  = False          # True = sirf ek frame check
STILL_FRAME = 1200           # race ke baad ka frame (shuru ke frames me bars dikhte nahi)
# ======================

PROJECT = "/content/my_project"
ROOT    = f"{PROJECT}/src/Root.jsx"

# ---------- 1. Checks ----------
if not os.path.exists(ROOT):
    raise SystemExit("❌ Root.jsx nahi mili. Pehle cell 2 chalayein.")
code = open(ROOT, encoding="utf-8").read()
if not code.strip():
    raise SystemExit("❌ Root.jsx khali hai.")
if not re.search(r"export\s+(?:const|let|function)\s+Root\b|export\s*\{[^}]*\bRoot\b", code):
    raise SystemExit("❌ Root.jsx me `export const Root` nahi mila. index.jsx isi ko import karta hai.")

# ---------- 2. Auto-fix: registerRoot(...) hatao ('called more than once' error se bachne ke liye) ----------
fixed = re.sub(r"^\s*registerRoot\(\s*\w+\s*\)\s*;?\s*$", "", code, flags=re.M)
if fixed != code:
    shutil.copy(ROOT, ROOT + ".bak")
    open(ROOT, "w", encoding="utf-8").write(fixed)
    code = fixed
    print("🛠️  registerRoot(...) line hata di (backup: Root.jsx.bak)")

# ---------- 3. Composition ID ----------
ids = re.findall(r"<Composition\b(?:(?!/>)[\s\S])*?\bid\s*=\s*[\"']([^\"']+)[\"']", code)
print("Mili hui compositions:", ids)
if not COMP_ID:
    if not ids:
        raise SystemExit("❌ Koi <Composition id=...> nahi mili. COMP_ID khud bharein.")
    COMP_ID = ids[-1]
    if len(ids) > 1:
        print(f"⚠️ Ek se zyada compositions mili, last wali use hogi: {COMP_ID}")
elif ids and COMP_ID not in ids:
    raise SystemExit(f"❌ COMP_ID '{COMP_ID}' code me nahi hai. Available: {ids}")
print("🎯 Render hoga:", COMP_ID)

# ---------- 4. Render ----------
os.makedirs(f"{PROJECT}/out", exist_ok=True)
shutil.rmtree(f"{PROJECT}/node_modules/.cache", ignore_errors=True)   # purana bundle cache hatao

if ONLY_STILL:
    out = f"{PROJECT}/out/check.png"
    cmd = ["npx", "remotion", "still", "src/index.jsx", COMP_ID, out, f"--frame={STILL_FRAME}"]
else:
    out = f"{PROJECT}/out/{OUTPUT_NAME}"
    cmd = ["npx", "remotion", "render", "src/index.jsx", COMP_ID, out,
           "--concurrency=2", "--gl=swiftshader", "--pixel-format=yuv420p", "--jpeg-quality=85"]

ansi = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
prog = re.compile(r"(Rendered|Encoded|Rendering|Encoding)\D*?(\d+)/(\d+)")
last = {}
p = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for raw in p.stdout:
    for line in raw.replace("\r", "\n").split("\n"):
        line = ansi.sub("", line).strip()
        if not line:
            continue
        m = prog.search(line)
        if m:                                    # progress: har 10% par ek line
            kind, a, b = m.group(1), int(m.group(2)), int(m.group(3))
            step = (a * 10) // max(b, 1)
            if last.get(kind) != step:
                last[kind] = step
                print(f"{kind}: {a}/{b}")
        elif line.startswith(("Bundling", "Bundled", "Getting Headless Shell")):
            continue                             # bundle/download ka shor skip
        else:
            print(line)
p.wait()

# ---------- 5. Download ----------
if p.returncode == 0 and os.path.exists(out):
    print(f"\n✅ Ready: {out} ({os.path.getsize(out)/1e6:.1f} MB)")
    from google.colab import files
    files.download(out)
else:
    print("\n❌ Render fail hua. Upar ka error padhein ya mujhe paste karein.")